# Финальная аналитика инвестиционного портфеля

## Структура проекта
- **.py файлы** — реализации каждого блока заданий
- **этот ноутбук** — вся аналитика, вспомогательные функции и выводы

## Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Безрисковая ставка (ставка ЦБ РФ ~16% годовых в 2024)
RISK_FREE_RATE = 0.16 / 252  # Дневная ставка (252 торговых дня в году)

print(f"Безрисковая ставка: {RISK_FREE_RATE*252*100:.2f}% годовых")

---

# БЛОК 1: ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ

## Загрузка и обработка данных

In [ ]:
def load_data(filepath):
    """Загрузка данных из CSV файла с учетом особенностей формата"""
    df = pd.read_csv(filepath, sep=';', decimal=',', parse_dates=['date'], dayfirst=True)
    df.set_index('date', inplace=True)

    # Удаляем лишние пробелы в названиях столбцов
    df.columns = df.columns.str.strip()

    # Удаляем акции с большим количеством пропусков
    missing_pct = df.isnull().sum() / len(df)
    valid_stocks = missing_pct[missing_pct < 0.1].index.tolist()
    df = df[valid_stocks]

    print(f"Загружено данных: {df.shape[0]} дней, {df.shape[1]} акций")
    print(f"Период: {df.index.min()} - {df.index.max()}")
    return df

In [ ]:
def calculate_returns(prices_df):
    """Расчет логарифмических доходностей"""
    returns = np.log(prices_df / prices_df.shift(1)).dropna()

    # Фильтрация экстремальных значений (возможные выбросы)
    for col in returns.columns:
        q1 = returns[col].quantile(0.01)
        q99 = returns[col].quantile(0.99)
        returns[col] = returns[col].clip(lower=q1, upper=q99)

    print(f"Рассчитано доходностей: {returns.shape}")
    return returns

In [ ]:
def estimate_parameters(returns_df):
    """Оценка ожидаемых доходностей и ковариационной матрицы"""
    # Ожидаемые доходности (средние дневные доходности)
    expected_returns = returns_df.mean()

    # Годовые доходности
    annual_returns = expected_returns * 252

    # Ковариационная матрица (дневная)
    cov_matrix = returns_df.cov()

    # Годовая ковариационная матрица
    annual_cov = cov_matrix * 252

    print(f"\nОжидаемые годовые доходности (топ-10):")
    print(annual_returns.sort_values(ascending=False).head(10))

    return expected_returns, annual_returns, cov_matrix, annual_cov

## Портфельные характеристики и оптимизация

In [ ]:
def portfolio_performance(weights, expected_returns, cov_matrix):
    """Расчет характеристик портфеля"""
    returns = np.dot(weights, expected_returns)
    volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, volatility

In [ ]:
def negative_sharpe_ratio(weights, expected_returns, cov_matrix, risk_free_rate):
    """Отрицательный коэффициент Шарпа для минимизации"""
    returns, volatility = portfolio_performance(weights, expected_returns, cov_matrix)
    sharpe = (returns - risk_free_rate) / volatility
    return -sharpe

In [ ]:
def optimize_portfolio(expected_returns, cov_matrix, risk_free_rate,
                      method='min_variance', target_return=None):
    """
    Оптимизация портфеля

    Параметры:
    - method: 'min_variance', 'max_sharpe', 'target_return', 'equal_weight'
    - target_return: целевая доходность (для method='target_return')
    """
    n_assets = len(expected_returns)
    args = (expected_returns, cov_matrix, risk_free_rate)

    # Начальные веса (равные)
    weights = np.ones(n_assets) / n_assets

    # Ограничения: веса неотрицательны и сумма равна 1
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    bounds = tuple((0, 1) for _ in range(n_assets))

    if method == 'equal_weight':
        weights = np.ones(n_assets) / n_assets
        return weights

    if method == 'min_variance':
        # Минимизация дисперсии
        def objective(w):
            returns, volatility = portfolio_performance(w, expected_returns, cov_matrix)
            return volatility ** 2

        # Аналитическое решение для портфеля минимальной дисперсии
        # w = Σ^(-1) * 1 / (1^T * Σ^(-1) * 1)
        try:
            inv_cov = np.linalg.pinv(cov_matrix.values)
            ones = np.ones(len(expected_returns))
            weights = inv_cov @ ones / (ones @ inv_cov @ ones)
            # Ограничиваем неотрицательность
            weights = np.maximum(weights, 0)
            weights = weights / weights.sum()
            return weights
        except Exception as e:
            # Fallback к численной оптимизации
            for opt_method in ['SLSQP', 'L-BFGS-B']:
                try:
                    constraints_to_use = constraints if opt_method == 'SLSQP' else None
                    result = minimize(objective, weights, method=opt_method,
                                     bounds=bounds, constraints=constraints_to_use)
                    if result.success:
                        return result.x
                except:
                    continue
            return weights

    elif method == 'max_sharpe':
        result = minimize(negative_sharpe_ratio, weights, args=args,
                         method='SLSQP', bounds=bounds, constraints=constraints)

    elif method == 'target_return':
        constraints = (
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w: portfolio_performance(w, expected_returns, cov_matrix)[0] - target_return}
        )
        def objective(w):
            returns, volatility = portfolio_performance(w, expected_returns, cov_matrix)
            return volatility ** 2
        result = minimize(objective, weights, method='SLSQP',
                         bounds=bounds, constraints=constraints)

    else:
        raise ValueError(f"Неизвестный метод оптимизации: {method}")

    return result.x

In [ ]:
def annualize_metrics(daily_return, daily_volatility):
    """Перевод дневных показателей в годовые"""
    annual_return = daily_return * 252
    annual_volatility = daily_volatility * np.sqrt(252)
    return annual_return, annual_volatility

## Визуализация

In [ ]:
def plot_efficient_frontier(expected_returns, cov_matrix, risk_free_rate,
                            min_var_weights, max_sharpe_weights,
                            save_path='efficient_frontier.png'):
    """Построение эффективной границы"""

    n_points = 100

    # Диапазон ожидаемых доходностей
    min_var_return, min_var_vol = portfolio_performance(min_var_weights, expected_returns, cov_matrix)
    max_sharpe_return, max_sharpe_vol = portfolio_performance(max_sharpe_weights, expected_returns, cov_matrix)

    min_ret = min_var_return * 0.9
    max_ret = max_sharpe_return * 1.2

    target_returns = np.linspace(min_ret, max_ret, n_points)
    frontier_returns = []
    frontier_volatilities = []

    for target_return in target_returns:
        try:
            w = optimize_portfolio(expected_returns, cov_matrix, risk_free_rate,
                                   method='target_return', target_return=target_return)
            ret, vol = portfolio_performance(w, expected_returns, cov_matrix)
            frontier_returns.append(ret)
            frontier_volatilities.append(vol)
        except:
            continue

    frontier_returns = np.array(frontier_returns)
    frontier_volatilities = np.array(frontier_volatilities)

    # Перевод в годовые показатели
    frontier_annual_returns = frontier_returns * 252 * 100
    frontier_annual_vols = frontier_volatilities * np.sqrt(252) * 100

    min_var_annual_ret, min_var_annual_vol = annualize_metrics(min_var_return, min_var_vol)
    max_sharpe_annual_ret, max_sharpe_annual_vol = annualize_metrics(max_sharpe_return, max_sharpe_vol)
    rf_annual = risk_free_rate * 252 * 100

    # Равнонагруженный портфель
    equal_weights = np.ones(len(expected_returns)) / len(expected_returns)
    equal_ret, equal_vol = portfolio_performance(equal_weights, expected_returns, cov_matrix)
    equal_annual_ret, equal_annual_vol = annualize_metrics(equal_ret, equal_vol)

    # Построение графика
    fig, ax = plt.subplots(figsize=(12, 8))

    # Эффективная граница
    ax.plot(frontier_annual_vols, frontier_annual_returns, 'b-', linewidth=2, label='Efficient Frontier')

    # CML (Capital Market Line)
    if frontier_annual_vols.size > 0:
        slope = (max_sharpe_annual_ret - rf_annual) / max_sharpe_annual_vol
        cml_vols = np.linspace(0, max(frontier_annual_vols) * 1.2, 100)
        cml_returns = rf_annual + slope * cml_vols
        ax.plot(cml_vols, cml_returns, 'r--', linewidth=1.5, label='CML (Capital Market Line)')

    # Портфели
    ax.scatter(min_var_annual_vol * 100, min_var_annual_ret, c='green', s=150, marker='s',
              label='Min Variance', zorder=5, edgecolors='black', linewidth=2)
    ax.scatter(max_sharpe_annual_vol * 100, max_sharpe_annual_ret, c='orange', s=150, marker='*',
              label='Max Sharpe', zorder=5, edgecolors='black', linewidth=2)
    ax.scatter(equal_annual_vol * 100, equal_annual_ret, c='purple', s=150, marker='^',
              label='Equal Weight', zorder=5, edgecolors='black', linewidth=2)

    # Безрисковый актив
    ax.scatter(0, rf_annual, c='black', s=150, marker='d',
              label=f'Risk-Free ({rf_annual:.2f}%)', zorder=5, edgecolors='gray', linewidth=2)

    ax.set_xlabel('Annual Volatility (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Annual Return (%)', fontsize=12, fontweight='bold')
    ax.set_title('Efficient Frontier', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\nГрафик эффективной границы сохранен: {save_path}")
    plt.close()

    return frontier_returns, frontier_volatilities

In [ ]:
def plot_portfolio_weights(tickers, weights_dict, save_path='portfolio_weights.png'):
    """Визуализация весов портфелей"""

    # Берем только топ-20 акций по весам в портфеле с макс. Шарпом
    max_sharpe_weights = weights_dict['max_sharpe']
    top_indices = np.argsort(max_sharpe_weights)[-20:][::-1]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    for idx, (name, weights) in enumerate(weights_dict.items()):
        ax = axes[idx]

        # Сортируем веса по убыванию
        sorted_indices = np.argsort(weights)[::-1]

        # Для каждого портфеля берем топ-15
        top_n = 15
        ax_indices = sorted_indices[:top_n]
        other_sum = weights[sorted_indices[top_n:]].sum()

        labels = list(tickers[ax_indices]) + ['Others']
        values = list(weights[ax_indices]) + [other_sum]

        # Фильтруем нулевые значения
        values_array = np.array(values)
        mask = values_array > 0.001
        labels = np.array(labels)[mask]
        values = values_array[mask]

        colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
        ax.barh(range(len(labels)), values, color=colors, edgecolor='black', alpha=0.8)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlabel('Weight', fontsize=11, fontweight='bold')
        ax.set_title(f'{name.upper()}', fontsize=12, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3)

        # Добавляем значения на столбцы
        for i, v in enumerate(values):
            ax.text(v + 0.01, i, f'{v*100:.1f}%', va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"График весов портфелей сохранен: {save_path}")
    plt.close()

---

# БЛОК 2: ЗАГРУЗКА ДАННЫХ

In [ ]:
# Загрузка данных
prices_df = load_data('../data/prices_moex_new.csv')

# Расчет доходностей
returns_df = calculate_returns(prices_df)

# Оценка параметров
expected_returns, annual_returns, cov_matrix, annual_cov = estimate_parameters(returns_df)

---

# БЛОК 3: ЗАДАЧА 2 - Расчет показателей эффективности портфелей

In [ ]:
# Импорт функций задач
import importlib.util
spec = importlib.util.spec_from_file_location("tasks_2_3", "tasks_2_3.py")
tasks_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tasks_module)

# Выполнение задачи 2
results_df, weights_dict = tasks_module.task_2(returns_df, expected_returns, cov_matrix, RISK_FREE_RATE)

### Анализ результатов задачи 2

In [ ]:
# Отображение результатов
print("\nСравнение портфелей:")
display(results_df.round(4))

---

# БЛОК 4: ЗАДАЧА 3 - Анализ динамики портфелей во времени

In [ ]:
# Выполнение задачи 3
stability_df, weights_history, dates_history = tasks_module.task_3(
    prices_df, window_years=1, step_months=6, risk_free_rate=RISK_FREE_RATE
)

### Анализ результатов задачи 3

In [ ]:
# Отображение метрик стабильности
print("\nМетрики стабильности портфелей:")
display(stability_df.round(4))

---

# БЛОК 5: ВЫВОДЫ

## Основные выводы

### Задача 2: Эффективные портфели
- **Портфель минимальной дисперсии**: 
  - Минимальный риск при допустимой доходности
  - Подходит для консервативных инвесторов

- **Портфель максимального Шарпа**:
  - Оптимальное соотношение риск-доходность
  - Максимальная премия за риск

- **Равнонагруженный портфель**:
  - Простейшая стратегия (1/n)
  - Может быть эффективной при ограничениях

### Задача 3: Стабильность во времени
- **Turnover (оборачиваемость)**: чем выше, тем больше транзакционных издержек
- **Количество активов**: диверсификация vs концентрация
- **Коэффициент вариации**: стабильность весов во времени

### Рекомендации
- При высокой волатильности рынка предпочитать портфель минимальной дисперсии
- При наличии транзакционных издержек учитывать turnover
- Регулярно ребалансировать портфель

In [ ]:
print("="*80)
print("АНАЛИЗ ЗАВЕРШЕН")
print("Созданные файлы:")
print("  - efficient_frontier.png")
print("  - portfolio_weights.png")
print("  - rebalancing_comparison.png")
print("  - portfolio_weights_heatmap.png")
print("="*80)